In [ ]:
import os 
from dotenv import load_dotenv
load_dotenv()

from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langgraph.types import interrupt
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import tools_condition, ToolNode
from langchain.messages import HumanMessage, SystemMessage
from langgraph.types import Command
from langgraph.checkpoint.memory import InMemorySaver
from typing_extensions import TypedDict, Annotated

config = { "configurable": { "thread_id": "1" }}
checkpointer = InMemorySaver()
llm = init_chat_model("groq:openai/gpt-oss-20b")

#Tools

@tool
def Sendemail(permission: str) -> str:
    """Send email after user approval."""

    approved = interrupt("Do you approve to send email?")

    if approved["decision"] == "yes":
        return "Email approved. Ready to send."

    return "Email rejected."

###llm with tool
llmWithTools = llm.bind_tools([Sendemail])


### STATE
class State(TypedDict):
    messages: Annotated[list, add_messages]
    approved: bool


#Node
def chatbot(state: State):
    response = llmWithTools.invoke([
        SystemMessage(content="""
        First collect all required information from the user.
        Then prepare a professional email draft.
        Show the draft to the user and ask for confirmation.
        Only after confirmation, call the Sendemail tool.
        """),
        *state["messages"]
    ])

    return {"messages": [response]}



graph_builder = StateGraph(State)

graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", ToolNode([Sendemail]))

graph_builder.add_edge(START, "chatbot")
graph_builder.add_conditional_edges("chatbot", tools_condition)
graph_builder.add_edge("tools", "chatbot")

graph = graph_builder.compile(checkpointer=checkpointer)
# response = graph.stream({"messages": [HumanMessage(content="Hello!")]}, config=config, stream_mode="values")
# print(response)
# for response in graph.stream({"messages": [HumanMessage(content="Hello!, my name is asjiad")]}, config=config, stream_mode="values"):
#     print(response)

for response in graph.stream(
    {"messages": [HumanMessage(content="""
Write an urgent leave request email.

Recipient: HR Manager
Email: asjadfarooq4@gmail.com
Reason: Family emergency
Start date: 2026-11-10
End date: 2026-11-12
Additional details: I will complete pending tasks before leaving.

Prepare a professional email, show me the draft,
and ask for my approval before sending.
""")]},
    config=config,
    stream_mode="values"
):
    pass

if response.get("__interrupt__"):
    print("Human approval required:", response["__interrupt__"])


response = graph.invoke(
    Command(
        resume={
            "decision": "yes"
        }  
    ),
    config=config
)

print(response["messages"][-1].content)


Human approval required: (Interrupt(value='Do you approve to send email?', id='edf607c5988f3a7beef64df521928027', response_schema=None),)
**Subject:** Urgent Leave Request – Family Emergency  

Dear HR Manager,

I hope this message finds you well. I am writing to request urgent leave due to a family emergency. I would like to take leave from **10 November 2026** to **12 November 2026**.

To ensure a smooth workflow during my absence, I will complete all pending tasks and hand over any critical responsibilities before my departure. I will also remain reachable for urgent queries if needed.

Thank you for your understanding and prompt consideration.

Kind regards,  
[Your Name]  
[Your Position]  
[Your Contact Information]

---

Please review the draft above. Let me know if you approve or if any adjustments are needed before I send it.
